# v16.7 — uji konfirmasi S1 (D-1 + episode AF ≥ 30 s) di wearable kontinu (Icentia11k) dan uji stres derau (nstdb)

Rancangan dikunci sebelum sinyal dibaca: [RANCANGAN.md](RANCANGAN.md) (sha256 di `../logs/v16.7_praregistrasi.log`). Logika di
`konfirmasi.py`; aturan/metrik/bootstrap dari v16.5–v16.6, rantai prediksi dari v16.4.

In [1]:
import json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import konfirmasi as K
C = K.C
import sys; sys.path[:0] = ["../v11_model_tim"]
from bersama import evaluasi as Ev
t0 = time.time()
R = {}
for ds in ("Icentia", "nstdb", "mitdb"):
    K.jalankan(ds)                    # dilewati bila cache per rekaman lengkap
D, info = {}, {}
for ds in ("Icentia", "nstdb", "mitdb"):
    D[ds], info[ds] = K.muat(ds)
R["data"] = info
display(pd.DataFrame(info).T)
ic = D["Icentia"]
for g in ("AF", "non-AF"):
    m = ic["grup"] == g
    print(g, "pasien", np.unique(ic["unit"][m]).size, "window", int(m.sum()), "window AF", int(ic["y2"][m].sum()), "jam", round(m.sum() * 5 / 3600, 1))

,rekaman,kosong,window_lolos,dibuang_label,window,window_AF,unit,jam
Icentia,600.0,0.0,472993.0,88557.0,384436.0,156995.0,600.0,533.938889
nstdb,12.0,0.0,4156.0,0.0,4156.0,0.0,2.0,5.772222
mitdb,2.0,0.0,720.0,0.0,720.0,0.0,2.0,1.000000


AF pasien 300 window 176651 window AF 156995 jam 245.3
non-AF pasien 300 window 207785 window AF 0 jam 288.6


## 1. Icentia: target, H1, H2

In [2]:
KOL = ("palsu_per_jam", "se_episode_30", "n_ref_30", "se_episode", "ppv_episode", "se_durasi", "ppv_durasi", "jam")
dn, da = K.bagian(ic, ic["grup"] == "non-AF"), K.bagian(ic, ic["grup"] == "AF")
H = {}
for nama, d in (("non-AF", dn), ("AF", da), ("semua", ic)):
    H[nama] = {s: C.nilai(d, af, 30, 0) for s, af in K.sistem(d).items()}
R["icentia"] = {g: {s: {k: m[k] for k in KOL} for s, m in h.items()} for g, h in H.items()}
for g in H:
    print("==", g); display(pd.DataFrame(R["icentia"][g]).T.round(3))
S1, RR = "S1 = D-1 + 30 s", "R = v3.1 + 30 s"
h1 = C.boot_rasio(H["non-AF"][S1]["palsu_unit"], H["non-AF"][RR]["palsu_unit"])
h2 = C.boot_rasio(H["AF"][S1]["se_unit"], H["AF"][RR]["se_unit"])
ci_se = C.boot_rasio(H["AF"][S1]["se_unit"], H["AF"][S1]["se_unit"])["ci_a"]
ci_p = C.boot_rasio(H["non-AF"][S1]["palsu_unit"], H["non-AF"][S1]["palsu_unit"])["ci_a"]
k = dict(target_se=bool(H["AF"][S1]["se_episode_30"] >= K.TARGET_SE), target_palsu=bool(H["non-AF"][S1]["palsu_per_jam"] <= K.TARGET_PALSU),
         H1=bool(h1["p_kurang"] >= 0.95), H2=bool(h2["p_lebih"] > 0.05))
R.update(H1=h1, H2=h2, ci=dict(se=ci_se, palsu=ci_p))
print(f"Se episode S1 {H['AF'][S1]['se_episode_30']:.3f} CI {np.round(ci_se, 3)}; palsu/jam non-AF {H['non-AF'][S1]['palsu_per_jam']:.3f} CI {np.round(ci_p, 3)}")
print("H1 (palsu/jam non-AF S1 − R):", h1); print("H2 (Se episode S1 − R):", h2)

== non-AF


,palsu_per_jam,se_episode_30,n_ref_30,se_episode,ppv_episode,se_durasi,ppv_durasi,jam
S1 = D-1 + 30 s,0.496,NaN,0.0,NaN,0.0,NaN,0.0,288.59
R = v3.1 + 30 s,2.332,NaN,0.0,NaN,0.0,NaN,0.0,288.59
S3 = (D-1 atau v3.1) + 30 s,2.845,NaN,0.0,NaN,0.0,NaN,0.0,288.59


== AF


,palsu_per_jam,se_episode_30,n_ref_30,se_episode,ppv_episode,se_durasi,ppv_durasi,jam
S1 = D-1 + 30 s,0.029,0.940,3975.0,0.487,0.999,0.856,1.000,245.349
R = v3.1 + 30 s,0.395,0.899,3975.0,0.466,0.984,0.813,0.994,245.349
S3 = (D-1 atau v3.1) + 30 s,0.420,0.970,3975.0,0.503,0.978,0.894,0.994,245.349


== semua


,palsu_per_jam,se_episode_30,n_ref_30,se_episode,ppv_episode,se_durasi,ppv_durasi,jam
S1 = D-1 + 30 s,0.281,0.940,3975.0,0.487,0.973,0.856,0.990,533.939
R = v3.1 + 30 s,1.442,0.899,3975.0,0.466,0.888,0.813,0.946,533.939
S3 = (D-1 atau v3.1) + 30 s,1.731,0.970,3975.0,0.503,0.831,0.894,0.940,533.939


Se episode S1 0.940 CI [0.916 0.96 ]; palsu/jam non-AF 0.496 CI [0.209 0.864]
H1 (palsu/jam non-AF S1 − R): {'delta': -1.8365137040691097, 'ci': [-2.662607522184967, -0.9954880760291915], 'p_lebih': 0.0, 'p_kurang': 1.0, 'titik_a': 0.49551218807902403, 'ci_a': [0.2092935514494717, 0.8643463844289884], 'titik_b': 2.332025892148134}
H2 (Se episode S1 − R): {'delta': 0.04125786163522005, 'ci': [0.021417230929443136, 0.06218676828821694], 'p_lebih': 1.0, 'p_kurang': 0.0, 'titik_a': 0.939874213836478, 'ci_a': [0.9161325046503611, 0.9602714632901875], 'titik_b': 0.8986163522012579}


## 2. nstdb (H3): alarm palsu per jam per SNR

In [3]:
nst = {}
for snr in ("bersih",) + K.SNR:
    d = D["mitdb"] if snr == "bersih" else K.bagian(D["nstdb"], D["nstdb"]["grup"] == snr)
    nst[snr] = {s: C.nilai(d, af, 30, 0)["palsu_per_jam"] for s, af in K.sistem(d).items()}
    nst[snr]["jam"] = float(d["start"].size * 5 / 3600)
R["nstdb"] = nst
display(pd.DataFrame(nst).T.round(3))
k["H3"] = bool(all(nst[s][S1] <= K.TARGET_PALSU for s in ("24", "18", "12")))
k["terkonfirmasi"] = all(k[x] for x in ("target_se", "target_palsu", "H1", "H2"))
R["keputusan"] = k
print(json.dumps(k))

,S1 = D-1 + 30 s,R = v3.1 + 30 s,S3 = (D-1 atau v3.1) + 30 s,jam
bersih,0.000,0.000,0.000,1.000
24,0.000,0.000,0.000,1.000
18,0.000,1.000,1.000,1.000
12,1.006,3.017,4.022,0.994
06,7.423,7.423,12.725,0.943
00,7.625,7.625,20.696,0.918
_6,5.455,10.909,20.727,0.917


{"target_se": true, "target_palsu": true, "H1": true, "H2": true, "H3": false, "terkonfirmasi": true}


## 3. Deskriptif: pita detak non-AF, per pasien

In [4]:
PITA = ((0, 100, "< 100"), (100, 110, "100–110"), (110, 120, "110–120"), (120, 999, "≥ 120"))
pita = {}
for s, af in K.sistem(ic).items():
    al = C.aturan_celah(ic["rec"], ic["start"], af, 5.0, 30, 0)
    non = ic["y2"] == 0
    pita[s] = {lab: dict(n=int((non & (ic["hr"] >= lo) & (ic["hr"] < hi)).sum()),
                         alarm=float(al[non & (ic["hr"] >= lo) & (ic["hr"] < hi)].mean()) if (non & (ic["hr"] >= lo) & (ic["hr"] < hi)).any() else None)
               for lo, hi, lab in PITA}
R["pita"] = pita
display(pd.DataFrame({s: {b: (f"{v['alarm']*100:.2f}% (n={v['n']})" if v["alarm"] is not None else "–") for b, v in p.items()} for s, p in pita.items()}).T)
pu = H["non-AF"][S1]["palsu_unit"]
R["nonaf_per_pasien_S1"] = {u: v[0] / v[1] for u, v in pu.items() if v[1] > 0}
x = np.array(list(R["nonaf_per_pasien_S1"].values()))
print(f"pasien non-AF dengan ≥ 1 alarm palsu: {(x > 0).sum()} dari {x.size}; > 1/jam: {(x > 1).sum()}; maks {x.max():.2f}")
su = H["AF"][S1]["se_unit"]
R["af_per_pasien_S1"] = su
print("pasien AF dengan episode ≥ 30 s yang terlewat semuanya:", sum(1 for v in su.values() if v[1] and v[0] == 0))

,< 100,100–110,110–120,≥ 120
S1 = D-1 + 30 s,0.58% (n=221064),0.32% (n=4010),0.11% (n=1828),0.74% (n=539)
R = v3.1 + 30 s,2.79% (n=221064),5.59% (n=4010),13.73% (n=1828),51.58% (n=539)
S3 = (D-1 atau v3.1) + 30 s,3.48% (n=221064),5.81% (n=4010),13.95% (n=1828),51.58% (n=539)


pasien non-AF dengan ≥ 1 alarm palsu: 26 dari 300; > 1/jam: 20; maks 32.69
pasien AF dengan episode ≥ 30 s yang terlewat semuanya: 5


## 4. Kesimpulan

In [5]:
a, n = R["icentia"]["AF"], R["icentia"]["non-AF"]
print(f"Icentia (wearable, {info['Icentia']['jam']:.0f} jam): Se episode AF ≥ 30 s S1 {a[S1]['se_episode_30']:.3f} vs R {a[RR]['se_episode_30']:.3f} "
      f"(S3 {a['S3 = (D-1 atau v3.1) + 30 s']['se_episode_30']:.3f}); alarm palsu/jam non-AF S1 {n[S1]['palsu_per_jam']:.3f} vs R {n[RR]['palsu_per_jam']:.3f} "
      f"(S3 {n['S3 = (D-1 atau v3.1) + 30 s']['palsu_per_jam']:.3f}).")
print(f"Target Se: {'lolos' if k['target_se'] else 'GAGAL'}; target alarm palsu: {'lolos' if k['target_palsu'] else 'GAGAL'}; H1 {'lolos' if k['H1'] else 'GAGAL'} "
      f"(P = {R['H1']['p_kurang']:.3f}); H2 {'lolos' if k['H2'] else 'GAGAL'} (P(Δ>0) = {R['H2']['p_lebih']:.3f}); H3 nstdb {'lolos' if k['H3'] else 'GAGAL'}.")
print("nstdb S1 palsu/jam per SNR:", {s: round(v[S1], 2) for s, v in R["nstdb"].items()})
print("Keputusan:", "S1 TERKONFIRMASI untuk wearable kontinu" if k["terkonfirmasi"] else "S1 belum terkonfirmasi untuk wearable (lihat kriteria yang gagal)")
R["lingkungan"] = Ev._lingkungan(); R["durasi_menit"] = (time.time() - t0) / 60
C.simpan_json(R, K.CACHE / "ringkasan.json"); print("tersimpan")

Icentia (wearable, 534 jam): Se episode AF ≥ 30 s S1 0.940 vs R 0.899 (S3 0.970); alarm palsu/jam non-AF S1 0.496 vs R 2.332 (S3 2.845).
Target Se: lolos; target alarm palsu: lolos; H1 lolos (P = 1.000); H2 lolos (P(Δ>0) = 1.000); H3 nstdb GAGAL.
nstdb S1 palsu/jam per SNR: {'bersih': 0.0, '24': 0.0, '18': 0.0, '12': 1.01, '06': 7.42, '00': 7.62, '_6': 5.45}
Keputusan: S1 TERKONFIRMASI untuk wearable kontinu
tersimpan
